# USA detector recovery: geometry, reviewed negatives, and recalibration

This bounded workflow uses the private, reviewed train/validation package prepared on 14 November 2026. It fixes padded-input box coordinates, resumes from the existing seed-1 Transformers weights, and keeps every new run separate from the selected production checkpoint. It does not publish a model or run access decisions.

The prepared COCO set has 249 train images (192 positive, 57 negative) and 79 validation images (41 positive, 38 negative). All March 23 target-camera clips stay in train; March 19 stays in validation. The 30 PKLot scenes remain validation-only. Test images are not in this archive. The target-camera validation negatives come from one recording day and do not establish generalization to other conditions.

The retained seed-1 diagnostic at confidence 0.60 found 40 TP, 10 FP and 1 FN after correcting coordinates. Seven FP were timestamp overlays on the eight target negatives. The explicit CH06 overlay filter removes those seven in that diagnostic, leaving precision 0.9302, below the 0.99 target. That filter was inspected on validation and must be treated as a chosen preprocessing setting, not an independent result. Some vehicle text still attracts detections. Retraining and a new independent recording remain necessary.


## 1. Install and clone the recovery revision

Select a GPU runtime before training. These dependencies come from the normal Python package registry. The three training seeds use the same starting checkpoint and separate run folders.


In [ ]:
import subprocess,sys
from pathlib import Path
packages=['transformers>=5.4,<6','accelerate>=1.2,<2','openvino>=2025.4,<2027','pycocotools>=2.0.8','pandas>=2.2,<4','PyYAML>=6,<7','Pillow','numpy']
subprocess.check_call([sys.executable,'-m','pip','install','-q',*packages])
REPO=Path('/content/SIRAP_USA_recovery_20261114')
BRANCH='fix/usa-camera-recovery-20261114'
if not REPO.exists():
    subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git',str(REPO)])
sys.path.insert(0,str(REPO))
CODE_REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
print('Recovery code revision:',CODE_REVISION)


## 2. Mount Drive and restore the prepared train/validation set

The archive is private and already contains the reviewed annotations. Its test partition is deliberately absent. Existing model files and the original reserved test archive stay in their current locations.


In [ ]:
from google.colab import drive
drive.mount('/content/drive_sirap')
import csv,json,shutil,zipfile
from datetime import datetime,timezone
PRIVATE=Path('/content/drive_sirap/MyDrive/SIRAP_USA_target_camera_negative_candidates_2026-11')
ARCHIVE=PRIVATE/'usa_full_scene_recovery_train_valid_20261114.zip'
DATA=Path('/content/SIRAP_recovery_data_20261114/recovery_coco_20261114')
if not (DATA/'recovery_dataset_manifest.json').is_file():
    DATA.parent.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(ARCHIVE) as archive:
        for member in archive.infolist():
            path=Path(member.filename)
            if path.is_absolute() or '..' in path.parts or ((member.external_attr>>16)&0o170000)==0o120000:
                raise ValueError('Unsafe archive path or link')
        archive.extractall(DATA.parent)
dataset_audit=json.loads((DATA/'recovery_dataset_manifest.json').read_text())
assert dataset_audit['test_included'] is False and not (DATA/'test').exists()
print(json.dumps(dataset_audit,indent=2))
BASE=Path('/content/drive_sirap/MyDrive/SIRAP_USA_ECUADOR/models/usa/detector/rtdetrv2_s/transformers')
if not BASE.is_dir():
    raise FileNotFoundError('The retained seed-1 Transformers checkpoint is required.')
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
LOCAL_RUNS=Path('/content/SIRAP_recovery_runs')/STAMP
DRIVE_RUNS=PRIVATE/('retraining_'+STAMP)


## 3. Retrain and preserve the three candidates

Enable `RUN_RETRAINING` only when ready to use the GPU. Training is capped at 50 epochs with patience 8, seeds 0/1/2. Each candidate starts from the retained seed-1 weights as domain adaptation. Model selection uses corrected-geometry validation mAP50–95. Candidate models and reports are copied to a new private run folder; this cell does not replace the canonical model.


In [ ]:
RUN_RETRAINING=False
SEEDS=[0,1,2]
EPOCHS=50
PATIENCE=8
candidates=[]
if RUN_RETRAINING:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Select a Colab GPU runtime before retraining.')
    from src.train import train_detector
    from src.evaluate import evaluate_detector
    DRIVE_RUNS.mkdir(parents=True,exist_ok=False)
    for seed in SEEDS:
        folder=LOCAL_RUNS/f'seed_{seed}'
        train_detector(DATA,country='usa',model_id=str(BASE),output_dir=folder,
                       run_dir=folder/'trainer',epochs=EPOCHS,early_stopping_patience=PATIENCE,
                       batch_size=8,learning_rate=5e-5,weight_decay=1e-4,seed=seed,force=False)
        metrics=evaluate_detector(DATA,country='usa',split='val',checkpoint=folder/'openvino',
                                  threshold=.001,result_path=folder/'validation_metrics.json')
        if metrics.get('map50_95') is None:
            raise RuntimeError('Candidate did not produce validation mAP50-95.')
        persistent=DRIVE_RUNS/f'seed_{seed}'
        for component in ('transformers','openvino'):
            shutil.copytree(folder/component,persistent/component)
        shutil.copy2(folder/'validation_metrics.json',persistent/'validation_metrics.json')
        candidates.append({'seed':seed,'map50_95':float(metrics['map50_95']),
                           'local_model':str(folder),'persistent_model':str(persistent)})
        (DRIVE_RUNS/'candidates.json').write_text(json.dumps(candidates,indent=2))
    selected=max(candidates,key=lambda r:r['map50_95'])
    (DRIVE_RUNS/'selected_checkpoint.json').write_text(json.dumps({'selected':selected,
        'code_revision':CODE_REVISION,'selection_split':'valid','starting_checkpoint':str(BASE),
        'test_evaluated':False,'dataset':dataset_audit},indent=2))
    print('Selected using validation:',selected)
else:
    print('Retraining is disabled. The retained selected checkpoint remains available.')


## 4. Calibrate the selected candidate on validation

This stage requires completed training above. The main threshold table uses the detector alone. A separate table applies the optional CH06 timestamp rule only to its known target-camera negative files; it is not a global USA rule. The report records the checkpoint hashes and code revision and remains private. It never enables access decisions. The 0.99 precision target and minimum support of 11 TP are inherited from the main notebook; confidence intervals and independent camera evidence remain necessary before operational use.


In [ ]:
from src.data_processing import load_image


def box_iou_xyxy(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    return inter / max(area_a + area_b - inter, 1e-9)

def collect_detection_records(split, max_images=None, transform=None):
    dataset = CocoPlateDataset(DATA_DIR, split, image_processor=None)
    source_images = dataset.images if max_images is None else dataset.images[:max_images]
    records = []
    for item in source_images:
        image_path = dataset._image_path(item["file_name"])
        image = load_image(image_path.read_bytes())
        if transform is not None:
            image = transform(image)
        gt_boxes = []
        for ann in dataset.annotations_by_image.get(int(item["id"]), []):
            x, y, w, h = map(float, ann["bbox"])
            gt_boxes.append((x, y, x + w, y + h))
        detections = detector.detect(image, max_detections=300)
        records.append({
            "image_id": int(item["id"]),
            "predictions": [(d.box, d.confidence) for d in detections],
            "ground_truth": gt_boxes,
        })
    return records

def score_detection_records(records, threshold, iou_threshold=0.50):
    tp = fp = fn = 0
    for record in records:
        unmatched = set(range(len(record["ground_truth"])))
        predictions = sorted(
            (item for item in record["predictions"] if item[1] >= threshold),
            key=lambda item: item[1], reverse=True,
        )
        for box, _ in predictions:
            best_index, best_iou = None, 0.0
            for index in unmatched:
                value = box_iou_xyxy(box, record["ground_truth"][index])
                if value > best_iou:
                    best_index, best_iou = index, value
            if best_index is not None and best_iou >= iou_threshold:
                tp += 1
                unmatched.remove(best_index)
            else:
                fp += 1
        fn += len(unmatched)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}



In [ ]:
RUN_RECALIBRATION=False
if RUN_RECALIBRATION:
    if not candidates:
        raise RuntimeError('Complete candidate training and selection first.')
    import hashlib,numpy as np,pandas as pd
    from src.train import CocoPlateDataset
    from src.model import PlateDetector
    from src.camera_regions import CH06_TIMESTAMP_PROFILE,filter_camera_overlays
    selected=max(candidates,key=lambda r:r['map50_95'])
    model=Path(selected['local_model'])/'openvino'
    DATA_DIR=str(DATA)
    detector=PlateDetector(checkpoint=model,conf=.001,fallback_checkpoint=None)
    records=collect_detection_records('val')
    dataset=CocoPlateDataset(DATA,'val',image_processor=None)
    target_names={f'neg_valid_031905_frame{n:03d}.jpg' for n in (2,5,10,12,18,23,28,34)}
    from src.model import Detection
    filtered=[]
    for item,record in zip(dataset.images,records):
        row=dict(record)
        if Path(item['file_name']).name in target_names:
            proposed=[Detection(box,confidence) for box,confidence in record['predictions']]
            kept,excluded=filter_camera_overlays(proposed,(item['height'],item['width']),CH06_TIMESTAMP_PROFILE)
            row['predictions']=[(d.box,d.confidence) for d in kept]
        filtered.append(row)
    tables={}
    for name,rows in [('detector_only',records),('known_ch06_overlay_filter',filtered)]:
        table=[{'threshold':round(float(t),2),**score_detection_records(rows,float(t))}
               for t in np.arange(.05,1,.05)]
        eligible=[r for r in table if r['precision']>=.99 and r['tp']>=11]
        best=max(table,key=lambda r:(r['f1'],-r['threshold']))
        tables[name]={'threshold_grid':table,'best_f1_candidate':best,'eligible_candidates':eligible}
    report={'country':'usa','selected':selected,'code_revision':CODE_REVISION,
            'postprocessing_version':'openvino_padded_boxes_v2',
            'checkpoint_sha256':hashlib.sha256((model/'openvino_model.bin').read_bytes()).hexdigest(),
            'validation_images':len(records),'tables':tables,'operational_threshold':None,
            'usable_for_access_decisions':False,'final_test_evaluated':False,
            'pending':'Independent target-camera test negatives and end-to-end operational validation'}
    output=DRIVE_RUNS/'recovery_validation_calibration.json'
    if output.exists():
        raise FileExistsError('This calibration report already exists.')
    output.write_text(json.dumps(report,indent=2),encoding='utf-8')
    for name,value in tables.items():
        print(name,'best F1 candidate:',value['best_f1_candidate'],'eligible:',len(value['eligible_candidates']))
    print('Private report:',output)
else:
    print('Recalibration is disabled until the candidate runs are complete.')


## 5. Independent final test remains pending

No test inference cell is provided in this recovery notebook. The currently reserved manual test has 42 positive images and zero negatives. March 18 has prior review exposure, March 19 is validation, and all March 23 recordings are training. Obtain a previously unexamined recording/day with reviewed negative scenes before freezing a final test protocol. Then freeze the selected checkpoint, code revision, camera preprocessing and threshold and use the main notebook's final-test workflow after its split-size and calibration requirements are met. Do not move a reviewed train/validation frame into test.

The candidate model is a diagnostic result until separate detector, OCR and decision evaluations establish the intended operating behavior. No Space publication, live camera session or automatic gate action occurs in this workflow.
